# F5-TTS Voice Cloning on Google Colab (T4)

Clones a voice from a short reference audio clip, then generates new speech in that cloned voice from any text you type.

### What this is (and isn't)
This notebook uses **F5-TTS** (`SWivid/F5-TTS`) — a diffusion-based voice cloning / text-to-speech model. It is *not* Coqui XTTS, even though an earlier version of this notebook was named for XTTS — the code inside was always F5-TTS.

**How it works:** give it ~5-15 seconds of clean reference audio of a voice (yours or anyone with rights to clone) + a short transcript of what that reference audio says, then type any new text — it generates that new text spoken in the cloned voice.

### Why the setup is simple (no conda env needed)
Unlike SadTalker, F5-TTS's dependencies install cleanly on Colab's current default Python (3.12) with the pre-installed torch build — no version conflicts, so no isolated environment is required here.

### Run order
| Step | Cell | What it does |
|---|---|---|
| 1 | Cell 1 | Confirms GPU is available |
| 2 | Cell 2 | Clones the F5-TTS repository |
| 3 | Cell 3 | Installs F5-TTS and all dependencies (~2-4 minutes) |
| 4A / 4B | Cell 4A **or** 4B | Pick one: 4A = Gradio web UI (recommended, drag-and-drop), 4B = command-line cloning (scriptable, for batch use) |

Before starting: **Runtime -> Change runtime type -> T4 GPU**.

### A note on responsible use
Only clone voices you have the right to use (your own voice, or with the speaker's explicit consent). Voice cloning can be misused for impersonation or fraud — don't use this to imitate someone without their permission.

---
## STEP 1 — Cell 1: Confirm GPU is connected
If this errors or shows no GPU, go to Runtime -> Change runtime type -> T4 GPU, then re-run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

---
## STEP 2 — Cell 2: Clone the F5-TTS repository

In [ ]:
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS

---
## STEP 3 — Cell 3: Install F5-TTS and dependencies
Takes ~2-4 minutes. Runs on Colab's default Python (no custom environment needed).

In [ ]:
%cd /content/F5-TTS
!pip install -q -e .
print("\nSetup complete. Continue to Step 4 (choose 4A or 4B below).")

---
## STEP 4A — Gradio Web UI (recommended, easiest)
Drag-and-drop your reference audio, type the reference transcript, type your new text, click Generate.

Running this cell starts a live server — that's normal, it will keep running. A public `*.gradio.live` link prints below; click it to open the UI in a new tab. Stop the cell when you're done to shut the server down.

Skip Cell 4B if you use this.

In [ ]:
%cd /content/F5-TTS
!f5-tts_infer-gradio --share

---
## STEP 4B — Command-line cloning (alternative to 4A)
Use this instead of the web UI if you want a scriptable, repeatable run (e.g. batch-generating many lines from the same cloned voice).

Upload your reference audio first, then edit `ref_text` and `gen_text` below to match your own content.

In [ ]:
from google.colab import files
import os

os.makedirs('/content/inputs', exist_ok=True)
print("Upload your REFERENCE AUDIO (5-15 sec, clean, minimal background noise, wav/mp3):")
ref = files.upload()
ref_name = list(ref.keys())[0]
ref_audio_path = '/content/inputs/' + ref_name
os.rename(ref_name, ref_audio_path)
print(f"\nReference audio saved to: {ref_audio_path}")

In [ ]:
%cd /content/F5-TTS

# ref_text  = the exact words spoken in your uploaded reference audio (must match what's actually said)
# gen_text  = the NEW text you want generated in the cloned voice
ref_text = "Type the exact transcript of your reference audio clip here."
gen_text = "Type the new sentence you want spoken in the cloned voice here."

!f5-tts_infer-cli \
  --ref_audio "{ref_audio_path}" \
  --ref_text "{ref_text}" \
  --gen_text "{gen_text}" \
  --output_dir /content/results

---
## STEP 5 — Download your generated audio (only needed if you used 4B)
If you used the Gradio UI (4A), download directly from the browser interface instead.

In [ ]:
import glob
import os
from google.colab import files

output_files = glob.glob('/content/results/**/*.wav', recursive=True)
output_files.sort(key=lambda x: -os.path.getmtime(x))

if output_files:
    latest = output_files[0]
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output audio found — check Cell 4B for errors.")

---
### Troubleshooting reference
| Symptom | Cause | Fix |
|---|---|---|
| `KeyboardInterrupt` after starting Gradio | Cell was manually stopped (not an actual error) | Just re-run the cell; this happens if you click Stop/interrupt the running server |
| Cloned voice sounds off / mispronounced words | `ref_text` doesn't exactly match what's said in the reference audio | Re-type `ref_text` to match the reference audio word-for-word |
| Cloned voice sounds robotic/noisy | Reference audio has background noise or is too short | Use a cleaner 5-15 sec clip with no music/noise in the background |
| CUDA out of memory | Rare on T4 for TTS, but possible with very long `gen_text` | Split long text into shorter chunks and generate separately |
| `pip install -e .` fails | Dependency conflict from a Colab image update | Restart runtime (Runtime -> Restart session) and re-run Cells 2-3 from scratch |
| Gradio link doesn't load | Public link can take 10-20 sec to activate after printing | Wait a few seconds and refresh the tab |

### Reminder
Only clone voices you have the right to use. Get explicit consent before cloning someone else's voice.